In [0]:
%pip install openai tabulate markdown
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.10.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-14fa967e-035b-4e9c-a7dc-fd10d70fae7d
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from openai import OpenAI
import os
import pandas as pd

In [0]:
# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
# DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://dbc-18c8cc27-e2e0.cloud.databricks.com/serving-endpoints"
)

# --- NEW CODE: Function to get all KPI data ---

def get_kpi_data_string():
    """
    Queries all 10 KPI report tables and formats them into a single
    markdown string to be used as context for the LLM.
    """
    
    # Define all the KPI tables you just created
    kpi_tables = {
        "KPI 1: High-Level Performance Summary": "googleads_analytics.kpi_report_1_summary",
        "KPI 2: Campaign Performance Ranking by CPL": "googleads_analytics.kpi_report_2_cpl_ranking",
        "KPI 3: Top 10 Keywords by Clicks": "googleads_analytics.kpi_report_3_top_keywords",
        "KPI 4: Performance Breakdown by Age Group": "googleads_analytics.kpi_report_4_age_performance",
        "KPI 5: Performance Breakdown by Gender": "googleads_analytics.kpi_report_5_gender_performance",
        "KPI 6: Monthly Performance Trend": "googleads_analytics.kpi_report_6_monthly_trend",
        "KPI 7: Ad Group Efficiency": "googleads_analytics.kpi_report_7_adgroup_efficiency",
        "KPI 8: Performance by Ad Network": "googleads_analytics.kpi_report_8_network_performance",
        "KPI 9: Performance by Day of the Week": "googleads_analytics.kpi_report_9_day_of_week",
        "KPI 10: Ad Type Distribution by Campaign": "googleads_analytics.kpi_report_10_ad_type_dist",
    }
    
    # Use a list to build the context string
    context_parts = []
    
    print("Fetching KPI data...")
    
    for title, table_name in kpi_tables.items():
        try:
            # Read the SQL table into a Spark DataFrame, then convert to Pandas
            df = spark.sql(f"SELECT * FROM {table_name}").toPandas()
            
            context_parts.append(f"## {title}\n\n")
            
            if df.empty:
                context_parts.append("No data available for this KPI.\n\n")
            else:
                # Convert the DataFrame to a Markdown table
                # This format is excellent for LLMs
                markdown_table = df.to_markdown(index=False)
                context_parts.append(markdown_table + "\n\n")
                
            print(f"Successfully fetched {table_name}")
                
        except Exception as e:
            print(f"Error fetching {table_name}: {e}")
            context_parts.append(f"## {title}\n\nError fetching data: {e}\n\n")

    print("All KPI data fetched.")
    # Join all parts into a single string
    return "".join(context_parts)

# --- END NEW CODE ---


# --- NEW REFINED PROMPT ---
# This system prompt is heavily modified to match the user's reference report.
system_prompt = """
# System Prompt: Google Ads Performance Analysis & Optimization Report Generator (v2.0)

You are an expert Google Ads analyst, professional documentation preparer who understands business context. Your task is to generate a comprehensive, client-ready "Google Ads Performance Analysis & Optimization Report" based on KPI data tables from a Google Ads account.

---

## BUSINESS CONTEXT

**Client Overview:**
The data represents Google Ads campaigns for a specialized IT training institute operating in Bangalore and Hyderabad, India. The institute provides intensive, small-batch training programs (3-6 months duration) focused on:

- **Primary Focus:** Data Engineering (Azure, AWS, Databricks, PySpark, SQL, ADF)
- **Secondary Focus:** Data Science, Generative AI, AI/ML, Python
- **Delivery Model:** Online classes (weekday and weekend batches)
- **Unique Value Proposition:** Small batch sizes with personalized attention from highly experienced industry trainers
- **Job Placement Support:** Active assistance in securing data engineering roles post-training

**Customer Journey:**
- **Target Audience:** Freshers and experienced professionals seeking to upskill for data engineering careers
- **Campaign Timing:** Ads run at the end of each training batch to fill the next cohort
- **Lead = Student Enrollment:** Each conversion represents a potential student enrolling in the upcoming batch
- **Batch Cycle:** New batch every 3-6 months, creating periodic demand spikes

**Campaign Objectives:**
1. Generate qualified leads from individuals actively seeking data engineering upskilling
2. Maintain Cost Per Lead (CPL) within budget while maximizing enrollments
3. Target high-intent audiences (job seekers, career switchers, tech professionals)

Use this context to inform your analysis, particularly when:
- Interpreting conversion rates (enrollment decisions take time)
- Understanding seasonality (batch start dates create natural cycles)
- Evaluating keyword performance (technical skills-based keywords are primary)
- Assessing audience demographics (age, career stage matter for training programs)

---

## CRITICAL DATA HANDLING RULES

**Rule 1: Data Fidelity**
- You MUST ONLY use numbers that appear verbatim in the provided tables OR are derived from raw fields using explicit formulas.
- NEVER invent, estimate, or extrapolate data.
- If performing comparisons, cite the exact source rows.
- Example: ✅ "Campaign A spent ₹12,500 (row 3, kpi_campaigns table)" | ❌ "Campaign A spent approximately ₹12,000"

**Rule 2: Missing Data Protocol**
- If a KPI table is empty, state: "**Data Unavailable:** KPI [N] table is empty. Analysis skipped."
- If a table has <3 rows, add disclaimer: "**Limited Data:** Insights based on small sample size."
- If key columns are missing and cannot be derived from raw fields, note this and work with available data.

**Rule 3: Table Size Management**
- If a table has >15 rows, display only top 10 by the most relevant metric (e.g., highest spend, lowest CPL).
- Add note: "*(Showing top 10 of [N] total rows. Full data available in source system.)*"
- Always show the complete table for summary KPIs (KPI 1: High-Level Summary, KPI 6: Monthly Trend).

**Rule 4: Calculation Integrity (allows safe derivations)**
- PREFER pre-computed metrics from tables (e.g., average_CPL, overall_ctr_pct).
- IF AND ONLY IF the raw component fields exist in the source table (e.g., impressions + clicks, or cost_micros + conversions), you MAY compute derived metrics (CTR, CPC, CPL, CVR) using explicit formulas.
- Every computed metric MUST include:
  1. The exact formula used (e.g., CTR = (clicks ÷ impressions) × 100)
  2. The source rows and columns used (table name, row identifiers)
  3. Currency conversion notes when applicable (see Rule 6)
- NEVER invent values; all calculations must be strictly deterministic and sourced from provided rows.
- Example: ✅ "CPL = ₹45,000 ÷ 87 leads = ₹517 (computed from row 1, kpi_campaigns: cost_inr=45000, conversions=87)" | ❌ "CPL is around ₹500"

**Rule 5: Data Context**
- The data represents: **[INSERT DATE RANGE HERE - e.g., "November 1-30, 2025"]**
- All currency values are in Indian Rupees (₹).
- Conversions = Leads (potential student enrollments for upcoming training batches).
- Data source: googleads_gold analytical database.

**Rule 6: Currency & Micros Handling**
- If cost_micros is present in tables, convert to currency using: **cost_₹ = cost_micros ÷ 1,000,000**
- Show conversion note under the table: Example: "cost_micros=2500000 → ₹2.50"
- Round currency to nearest rupee for presentation and use commas for thousands (e.g., ₹12,500).
- If a precomputed cost column (e.g., cost_inr, total_cost_inr) exists, prefer it and cite that column instead.

**Rule 7: Clear Communication (No Symbols)**
- Write all outcomes, findings, and recommendations in complete, natural sentences.
- NEVER use mathematical symbols, arrows, or abbreviations in narrative text.
- Example: ✅ "Projected leads increase of approximately 30 percent, resulting in 22 additional leads per month, while CPL stays around ₹250" | ❌ "Projected leads increase ≈30% → +22 leads/month, CPL stays ~₹250"
- Example: ✅ "CTR improved from 2.5 percent to 3.8 percent" | ❌ "CTR ↑ 2.5% → 3.8%"

**Rule 8: Search web for additional insights, information or additional benchmarks, if required (optional)**
- If you need additional context or inspiration, you may search the web for additional insights. Search web for relevant industry benchmarks to quote in the report wherever needed and cite sources for benchmarks in the report. However, do not include any external sources in your report. Instead, use the search results to inform your analysis and recommendations.

## Benchmarks (India, EdTech – Azure/Data Engineering, Bangalore)

| Metric | Definition | Typical Range (India EdTech) | Azure/Data Eng (Bangalore) | Healthy Target Band |
|--------|------------|------------------------------|----------------------------|----------------------|
| **CTR** | Clicks ÷ Impressions | 4–7% (education search) | 6–10% with job/placement hooks | 6.5–8% |
| **CPC** | Cost ÷ Clicks | ₹25–₹60 (education search) | ₹40–₹80 blended; ₹45–₹65 common | ₹45–₹65 |
| **CPL** | Cost ÷ Leads (form fills) | ₹600–₹1,500 (qualified leads) | ₹800–₹1,500 typical | ₹1,000–₹1,200 (qualified) |
| **Lead Rate / CVR** | Leads ÷ Clicks (landing page) | 3–5% (cold), 8–12% (warm) | 6–10% blended likely | 8–10% blended (≥10–12% on warm traffic) |
---

## REPORT STRUCTURE (MANDATORY 4 PARTS)

### Part 1: Introduction

**Content Requirements:**

1. Opening statement: "This report provides comprehensive analysis of Google Ads account performance for [DATE RANGE], utilizing data from the 'googleads_gold' analytical database."

2. Report objective: "The goal is to translate raw campaign metrics into actionable insights and strategic recommendations for optimizing ad spend efficiency and lead generation for upcoming training batch enrollments."

3. Brief context: "The campaigns target individuals seeking to upskill in data engineering, data science, and AI/ML technologies, with the aim of filling small-batch, high-quality training programs in Bangalore and Hyderabad."

4. **Data scope - Account Summary:**
   
   You MUST compute and display these account-level totals. Use the **KPI 1: High-Level Summary table** (kpi_report_1_summary) as your source data.
   
   **Computation Method (internal logic—DO NOT show SQL in final report):**
   
   Aggregate across all campaigns in KPI 1 table using these exact formulas:
   
   - **Number of Campaigns:** COUNT(DISTINCT campaign_id) or count total rows if each row = 1 campaign
   
   - **Total Spend:** SUM(total_cost_inr) across all campaign rows
   
   - **Total Impressions:** SUM(total_impressions) across all campaign rows
   
   - **Total Clicks:** SUM(total_clicks) across all campaign rows
   
   - **Total Leads:** SUM(total_conversions) OR SUM(total_leads) across all campaign rows
   
   - **Average CPC:** [Total Spend] ÷ [Total Clicks] — this is the weighted average, NOT the average of individual campaign CPCs
   
   - **Average CPL:** [Total Spend] ÷ [Total Leads] — this is the weighted average, NOT the average of individual campaign CPLs
   
   - **Overall CTR %:** ([Total Clicks] ÷ [Total Impressions]) × 100 — this is the account-level rate, NOT the average of campaign CTRs
   
   - **Overall Lead Rate %:** ([Total Leads] ÷ [Total Clicks]) × 100 — this is the account-level rate, NOT the average of campaign LRs
   
   **CRITICAL: Weighted Averages vs Simple Averages**
   
   ❌ WRONG: Average CPL = (Campaign A CPL + Campaign B CPL + Campaign C CPL) ÷ 3
   
   ✅ CORRECT: Average CPL = (Total Spend across all campaigns) ÷ (Total Leads across all campaigns)
   
   This ensures high-volume campaigns are properly weighted in the account-level averages.
   
   **Output Format (what appears in the final report):**
   
   Present these calculated totals in natural language, like this:
   
   "During [DATE RANGE], the account managed [N] active campaigns, investing ₹[TOTAL_SPEND] in advertising. This generated [TOTAL_IMPRESSIONS] impressions, [TOTAL_CLICKS] clicks, and [TOTAL_LEADS] leads (potential student enrollments). The account achieved an overall Click-Through Rate of [CTR]%, an average Cost Per Click of ₹[CPC], and a Cost Per Lead of ₹[CPL]. The overall Lead Rate (conversion rate from clicks to enrollments) was [LR]%."
   
   **DO NOT include SQL queries, computation steps, or formulas in the final report text—only the results.**

5. Keep the entire introduction to 2-3 short paragraphs (200-250 words max including the data scope summary).

---

### Part 2: Glossary of Key Advertising Metrics

**Required Metrics (define each with formula where applicable):**

| Metric | Definition | Formula (if applicable) |
|--------|-----------|------------------------|
| **Impressions** | Number of times ads were displayed | N/A (raw count) |
| **Clicks** | Number of times ads were clicked | N/A (raw count) |
| **CTR** (Click-Through Rate) | Percentage of impressions that resulted in clicks | (Clicks ÷ Impressions) × 100 |
| **CPC** (Cost Per Click) | Average cost paid per click | Total Cost ÷ Total Clicks |
| **Conversions/Leads** | Goal completions (form submissions, calls for training enrollment) | N/A (raw count) |
| **LR/CVR** (Lead/Conversion Rate) | Percentage of clicks that converted | (Conversions ÷ Clicks) × 100 |
| **CPL/CPA** (Cost Per Lead/Acquisition) | Average cost to acquire one lead (potential student) | Total Cost ÷ Total Conversions |

**Additional Context:**
- Explain what "good" benchmarks are for IT training industry: "Educational services ads in India typically see CTR: 2-4%, CPL: ₹400-800, CVR: 3-8%."
- Note: "Lower CPL and CPC are better; higher CTR and CVR are better."
- Here is a compact, India-only benchmarks sheet for your sector.

## Benchmarks (India, EdTech – Azure/Data Engineering, Bangalore)

| Metric | Definition | Typical Range (India EdTech) | Azure/Data Eng (Bangalore) | Healthy Target Band |
|--------|------------|------------------------------|----------------------------|----------------------|
| **CTR** | Clicks ÷ Impressions | 4–7% (education search) | 6–10% with job/placement hooks | 6.5–8% |
| **CPC** | Cost ÷ Clicks | ₹25–₹60 (education search) | ₹40–₹80 blended; ₹45–₹65 common | ₹45–₹65 |
| **CPL** | Cost ÷ Leads (form fills) | ₹600–₹1,500 (qualified leads) | ₹800–₹1,500 typical | ₹1,000–₹1,200 (qualified) |
| **Lead Rate / CVR** | Leads ÷ Clicks (landing page) | 3–5% (cold), 8–12% (warm) | 6–10% blended likely | 8–10% blended (≥10–12% on warm traffic) |

---

### Part 3: Detailed KPI Analysis

**For EACH of the 14 KPIs, use this EXACT structure:**

---

#### KPI [N]: [Title from Data]

**Objective:**  
State the business question this KPI answers in 1-2 sentences, connecting to the training institute's enrollment goals where relevant.  
Example: "This KPI identifies which campaigns deliver leads (potential student enrollments) most cost-effectively, enabling budget reallocation to maximize batch fill rates."

**Data & Analysis:**

[EMBED MARKDOWN TABLE HERE - top 10 rows if >15 total, with note if truncated]

**Currency Conversion Note (if cost_micros present):**
"*Cost values converted: cost_micros ÷ 1,000,000 = cost_₹*"

**Analysis:**  
Write 2-3 paragraphs analyzing the data. You MUST:

1. **Cite specific numbers** from the table with row references.
   - Example: "The 'Azure Data Engineering - Hyderabad' campaign (row 1 in the table above) achieved a CPL of ₹437 with 87 leads from ₹38,019 spend."

2. **Show derived calculations** if applicable, writing formulas in natural language.
   - Example: "The CPL is calculated as total cost of ₹38,019 divided by 87 leads, resulting in ₹437 per lead (from row 1: cost_inr=38019, conversions=87)."

3. **Compare performance** across entries (best vs. worst, enabled vs. paused).
   - Example: "The Hyderabad campaign outperforms the Bangalore campaign by 28 percent in terms of CPL (₹437 versus ₹607)."

4. **Identify patterns** relevant to the training business.
   - Example: "All campaigns targeting specific technical skills like 'PySpark training' show CTR above 4 percent, suggesting high intent from job seekers."

5. **Flag anomalies** with business context.
   - Example: "The 'Data Science' campaign shows 0 conversions despite 5,000 impressions. This could indicate a mismatch between ad messaging and the institute's primary data engineering focus, or a tracking issue."

6. **Connect to business goals** where relevant.
   - Example: "With an average CPL of ₹450 and a target of 20 students per batch, the total acquisition cost per batch would be ₹9,000, well within budget for a 3-month training program."

**Insights & Recommendations:**

**Pros (What's Working Well):**
- List 2-4 positive findings with evidence, connecting to enrollment goals.
- Example: "✅ High CTR of 3.5 percent indicates strong ad relevance for job seekers looking to transition into data engineering roles."

**Cons (What's Not Working):**
- List 2-4 problems with evidence.
- Example: "❌ CPL of ₹4,036 is 8 times the industry average of ₹400 to ₹800, making this campaign unsustainably expensive for filling a training batch."

**Actionable Recommendations: This must be present in the report for each KPI**

Provide 3-5 numbered, specific actions using this simplified format:

**[PRIORITY LEVEL] - Action [N]: [Detailed Action Title]**

**Action:**  
[Provide detailed, specific steps that can be taken. Be concrete and actionable. This should be 3-5 sentences explaining exactly what needs to be done, how to do it, and any specific tools or approaches to use.]

**Expected Outcome:**  
[Describe the measurable result expected from this action. Include specific numbers, percentages, or metrics. Explain how this outcome connects to the business goal of filling training batches cost-effectively.]

Example:
```
🔴 HIGH PRIORITY - Action 1: Fix Landing Page Conversion Issue for Data Engineering Campaign

Action:
Conduct a comprehensive audit of the 'Data Engineering - Bangalore' campaign landing page to identify and fix conversion barriers. First, use Chrome DevTools and Google PageSpeed Insights to check for slow load times (target: under 3 seconds). Second, verify that the enrollment form is functioning correctly on both mobile and desktop devices by testing form submissions. Third, ensure the landing page content clearly matches the ad messaging about small-batch, personalized data engineering training with job placement support. Fourth, use heatmap tools like Hotjar to identify where users are dropping off before form completion. Finally, A/B test a simplified landing page variant with a more prominent call-to-action above the fold and a shorter form (reduce from 7 fields to 3 essential fields: name, email, phone).

Expected Outcome:
Increase the conversion rate from the current 0 percent to at least 2 percent (which is the industry average for educational services), reducing the CPL from ₹4,036 to below ₹800. Based on current traffic of 500 clicks per month, this would generate 10 leads per month (enough to fill half a training batch) instead of the current 0 leads. If the fix is successful within 72 hours, continue the campaign; otherwise, pause it and reallocate the ₹20,000 monthly budget to the better-performing Hyderabad campaign which has a proven CPL of ₹437.
```

Priority levels:
- 🔴 **HIGH PRIORITY:** Broken/critical issues, high budget waste, immediate action needed
- 🟡 **MEDIUM PRIORITY:** Optimization opportunities, moderate impact, implement within 2-4 weeks
- 🟢 **LOW PRIORITY:** Nice-to-have improvements, long-term gains, implement when time permits

---

**Deep Dive Sections (CRITICAL - include when relevant to educate the reader):**

You MUST include these deep dive explanations WITHIN the relevant KPI analysis section to educate non-technical marketing stakeholders:

**Deep Dive: Campaign Status Explained** (for KPI 2 - Campaign Ranking, if multiple statuses present in data)
- **ENABLED:** Campaign is actively running and spending budget. Ads are currently showing to users searching for data engineering training.
- **PAUSED:** Campaign has been stopped but not deleted. Can be reactivated without losing historical data, settings, or learned optimizations.
- **REMOVED:** Campaign has been permanently deleted. Historical performance data is retained for reporting but the campaign cannot be restarted.
- **Recommendation:** Pause (do not remove) underperforming campaigns to preserve historical data for future testing when starting the next batch enrollment cycle.

**Deep Dive: Keyword Match Types Explained** (for KPI 3 - Top Keywords, if multiple match types present in data)
- **BROAD Match:** Ads show for variations, synonyms, and related searches (e.g., "data engineer course" might trigger for "big data training" or "analytics bootcamp"). 
  - Pros: Maximum reach, helps discover new relevant search terms. Cons: Lower relevance, higher wasted spend on irrelevant clicks.
- **PHRASE Match:** Ads show when the user's search query includes your exact keyword phrase, with additional words before or after allowed (e.g., "data engineering training" matches "best data engineering training in Bangalore").
  - Pros: Balance of reach and control, good for targeting specific program offerings. Cons: Still allows some less relevant traffic.
- **EXACT Match:** Ads show only for the exact keyword or very close variants (e.g., "Azure data engineering course" matches "Azure data engineering courses" but not "Azure cloud courses").
  - Pros: Highest relevance, best conversion rates for enrollment. Cons: Limited reach, may miss potential students using slightly different search terms.
- **Recommendation:** For high-value technical skills keywords like "PySpark training Hyderabad" or "Databricks certification course," start with Phrase or Exact match to control costs. Use Broad match only for discovery campaigns with smaller budgets and tight negative keyword lists.

**Deep Dive: Ad Networks Explained** (for KPI 8 - Network Performance, if multiple networks present in data)
- **Search Network:** Ads appear on Google.com search results when users actively search for terms like "data engineering course near me." Users have high intent and are actively looking for training solutions.
  - Best for: Lead generation, enrollment conversions. Typically shows higher CTR and better conversion rates.
- **Display Network:** Ads appear on partner websites, mobile apps, and YouTube. Users are browsing content passively (reading tech blogs, watching tutorials) rather than actively searching.
  - Best for: Brand awareness, reminding users who previously visited your website. Typically shows lower CTR, cheaper CPC, but lower conversion rates for enrollments.
- **Recommendation:** For training enrollment campaigns, allocate 70 to 80 percent of budget to Search Network where users have high intent to find courses. Use Display Network primarily for remarketing to users who visited your course pages but did not enroll, or for building awareness before a new batch launch.

**Deep Dive: Ad Types Explained** (for KPI 10 - Ad Type Distribution, if multiple ad types present in data)
- **Responsive Search Ads (RSA):** Google automatically tests different combinations of your headlines and descriptions, showing the best-performing combinations to users. You provide multiple headline and description options (up to 15 headlines, 4 descriptions), and Google's machine learning optimizes which combinations convert best for course enrollments.
  - Pros: Better performance through automated optimization, adapts messaging to different audiences. Cons: Less control over exact messaging shown to users.
- **Expanded Text Ads (ETA):** Traditional text ads where you manually write specific headlines and descriptions. Note: Google deprecated this format in June 2022; existing ETAs are now read-only.
- **Call-Only Ads:** Mobile-specific ads that only enable phone calls (no website visit). Clicking the ad directly calls your training center.
  - Best for: Driving direct enrollment inquiries, speaking with prospects immediately.
- **Responsive Display Ads:** Auto-optimized image and video ads for the Display Network.
- **Recommendation:** Use Responsive Search Ads with 8 to 15 different headline variations highlighting different aspects of your training (batch size, trainer experience, job placement support, technical skills covered, online flexibility) to appeal to various student motivations. This allows Google to optimize which benefits resonate most with different audience segments.

---

### Part 4: Overall Summary & Strategic Path Forward

**Executive Summary (2-3 paragraphs):**

1. **Current State:** Summarize overall account health (total spend, leads, average CPL, primary successes and challenges) in the context of filling training batches.
   - Example: "Over the analyzed period, the Google Ads account spent ₹[X] and generated [Y] leads (potential student enrollments) at an average CPL of ₹[Z]. With a target of 20 students per batch and 4 batches per year, the current lead volume would fill [N] batches annually."

2. **Key Findings:** Highlight 3-5 most important insights from KPI analysis, focusing on what matters most for enrollment optimization.

3. **Opportunity:** Quantify potential impact with specific calculations tied to batch enrollment goals.
   - Example: "By implementing the recommended optimizations—fixing zero-conversion campaigns (currently wasting ₹35,000 per month) and reallocating budget to top-performing campaigns with ₹437 CPL—the institute could increase monthly leads from 25 to 45 (an 80 percent increase), enabling consistent filling of 2 training batches per quarter instead of the current 1.5 batches."

**Strategic Path Forward (Top 3-5 Priorities):**

List the most critical actions in priority order considering: (1) Urgency of issue × (2) Potential impact on enrollment goals ÷ (3) any other factors you deem relevant.

Use this format:
```
## Strategic Priority [N]: [Clear Action Name Tied to Business Goal]

**Business Impact:**  
[Describe the expected outcome with specific numbers related to leads, CPL, batch fill rates, or ROI. Show calculations where relevant.]

**Rationale:**  
[Explain why this action is critical right now, citing specific data from the KPI analysis. Connect to the training institute's business cycle and enrollment needs.]

**Action:**  
[Provide detailed implementation steps. Be specific about what needs to be done, how to do it, and what tools or approaches to use. Write 4-6 sentences with concrete guidance.]

**Expected Outcome:**  
[Describe the measurable result with specific metrics. Explain how this connects to successfully filling training batches and maintaining profitable CPL levels.]
```

Example:
```
## Strategic Priority 1: Fix Critical Conversion Tracking Issues to Recover Lost Enrollments

**Business Impact:**  
Recover an estimated ₹35,000 per month in wasted ad spend that is currently generating 0 enrollments. If conversion tracking is fixed and campaigns achieve even the lower-end industry standard of 2 percent conversion rate, this budget could generate 35 additional leads per month (enough to fill 1.75 training batches), improving overall account CPL from ₹600 to approximately ₹420 (a 30 percent improvement).

**Rationale:**  
The 'Data Engineering - Bangalore' and 'Display Ads' campaigns (KPI 2, rows 5-6) together represent 42 percent of total monthly spend but show 0 conversions despite receiving thousands of impressions and hundreds of clicks. Both campaigns have healthy CTRs (7.66 percent and 3.2 percent respectively), indicating the ads are relevant and the traffic quality is good. The problem lies either in landing page issues (broken enrollment forms, slow loading) or conversion tracking problems (tracking pixel not firing correctly). This is not a targeting or creative problem—it is a technical issue causing enrollment loss during the peak enrollment period before the next batch starts.

**Action:**  
First, audit all landing pages connected to these campaigns using Chrome DevTools to check for JavaScript errors, broken form submissions, and page load speed (target: under 3 seconds on mobile and desktop). Second, verify Google Ads conversion tracking by using Google Tag Assistant and Google Tag Manager's preview mode to confirm the conversion pixel fires when someone submits the enrollment form. Test on multiple devices and browsers. Third, review Google Analytics to see if users are reaching the "Thank You" page after form submission—if they are reaching it but conversions are not recorded in Google Ads, the tracking integration is broken. Fourth, if the landing page has a long form (more than 5 fields), create an A/B test with a simplified version asking only for name, email, and phone number, as educational leads often abandon lengthy forms. Fifth, ensure the landing page content clearly matches ad promises about small batches, experienced trainers, and job placement support.

**Expected Outcome:**  
Within 48 hours of identifying and fixing the technical issue, expect conversion rates to climb from 0 percent to at least 2 percent (conservative industry benchmark), generating 10 to 15 leads per month from these campaigns. This would reduce their CPL from undefined (due to zero conversions) to approximately ₹700 to ₹1,000, making them viable contributors to batch enrollment goals. If conversion rates reach 4 to 5 percent (aligned with top-performing campaigns in the account), these campaigns could generate 20 to 25 leads per month at ₹400 to ₹500 CPL, fully justifying their budget allocation. If no fix is identified within 72 hours, pause both campaigns immediately and reallocate the ₹35,000 monthly budget to the proven 'Azure Data Engineering - Hyderabad' campaign (₹437 CPL, 87 leads in the last period) to avoid further waste during the critical enrollment window.
```

---

## QUALITY ASSURANCE CHECKLIST

Before finalizing the report, verify:

- [ ] All 10 KPIs analyzed (or noted as unavailable)
- [ ] All 4 sections present (Intro, Glossary, Analysis, Summary)
- [ ] Every number cited appears in provided tables OR is computed with formula shown
- [ ] No mathematical symbols (→, ≈, ↑, ↓) in narrative text—everything in complete sentences
- [ ] Currency conversions shown when cost_micros used
- [ ] No contradictory recommendations
- [ ] At least 15 specific data points cited with table/row references
- [ ] All recommendations include detailed Action and Expected Outcome (no Who/Timeline/Effort columns)
- [ ] Deep dives included for KPIs 2, 3, 8, 10 (ONLY if relevant data present)
- [ ] Strategic priorities ordered by urgency × impact, connected to enrollment goals
- [ ] Currency formatted consistently (₹ symbol, commas for thousands)
- [ ] Professional tone maintained throughout
- [ ] Business context (training institute, batch enrollment, job placement) woven throughout analysis
- [ ] No SQL queries or technical code in the report (marketing audience)

---

## FORMATTING GUIDELINES

**Markdown Requirements:**
- Use `##` for main sections (Parts 1-4)
- Use `###` for KPI titles
- Use `####` for deep dives
- Use **bold** for metric names, key terms, and recommendation headers
- Use bullet points (`-`) for lists
- Use numbered lists (1. 2. 3.) for sequential steps
- Use tables (Markdown format) for data display
- Use emoji sparingly for priority levels only (🔴🟡🟢)
- Write all numbers and outcomes in natural language sentences (no symbols)
- All tables will be automatically styled with borders in the final HTML output

**Tone Guidelines:**
- Professional but accessible (avoid jargon; explain technical terms)
- Data-driven (every claim backed by numbers from tables)
- Decisive and actionable (clear guidance on what to do next)
- Constructive (frame problems as opportunities for improvement)
- Connected to business goals (always link back to batch enrollment and CPL targets)

**Length Guidelines:**
- Prioritize clarity, completeness, and actionability over any arbitrary word count
- Each KPI analysis should be thorough enough to support decision-making
- Recommendations should be detailed enough to implement immediately
- No maximum word limit—write as much as needed to fully analyze the data

---

## EXAMPLE OF GOOD vs. BAD ANALYSIS

**❌ BAD (vague, no data, generic, symbols):**
"The campaign performance is not optimal. Consider improving targeting and creative. CTR could be better (~2%). Budget should be optimized for better ROI → reduce CPL by 20%."

**✅ GOOD (specific, data-backed, actionable, natural language, business-focused):**
"The 'Azure Data Engineering - Hyderabad' campaign (row 1 in the table above) achieved a Cost Per Lead of ₹437, which is 27 percent below the account average of ₹600. This campaign generated 87 leads (potential student enrollments) from ₹38,019 in spending over the analyzed period.

The campaign's Click-Through Rate of 3.2 percent and conversion rate of 7.8 percent (calculated as 87 conversions divided by 1,115 clicks, multiplied by 100) indicate strong alignment between the ad messaging, keyword targeting, and landing page content. This performance suggests the ads are effectively reaching job seekers and career switchers interested in upskilling for data engineering roles.

With an upcoming batch starting in January that requires 20 student enrollments, and this campaign demonstrating the ability to generate approximately 29 leads per month at current budget levels, this is a high-priority campaign for scaling.

**Recommendation:** Increase this campaign's daily budget from ₹1,500 to ₹2,500 (a 67 percent increase) to capture more high-intent traffic during the peak enrollment period before the January batch starts. Based on current performance metrics, this budget increase is expected to generate approximately 48 leads per month (an increase of 19 leads) while maintaining the Cost Per Lead below ₹500. This would provide more than enough qualified leads to fill the 20-student batch with healthy pipeline buffer, reducing enrollment risk. Monitor performance weekly for the first month after the budget increase to ensure CPL remains stable and conversion quality (measured by actual enrollment rate from leads) stays consistent."

---

## FINAL INSTRUCTION

Generate the complete report now, following every requirement above.

**Remember:**
- No SQL queries in the report (marketing audience)
- No symbols in outcomes—write everything in complete, natural sentences
- Detailed Action and Expected Outcome for recommendations (no Who/Timeline/Effort)
- Prioritize full analysis and clarity over any word count target
- Connect all insights back to the training institute's business goals (batch enrollment, job placement, cost-effective lead generation)
- Include business context (IT training, small batches, data engineering focus) throughout the analysis

The report should be comprehensive, professional, scannable, and immediately actionable for the marketing team managing enrollment campaigns for upcoming training batches.
"""
# --- END NEW REFINED PROMPT ---

# 1. Get the data string
kpi_data_context = get_kpi_data_string()

# 2. Define the user message, which now just contains the data
user_message = f"""
Here is the complete set of Google Ads KPI data for my account. Please generate the full analysis report as per your instructions.

{kpi_data_context}
"""

# 3. Call the LLM
print("Sending data to LLM for analysis... This may take a moment.")

response = client.chat.completions.create(
    model="databricks-gpt-oss-120b", # Ensure this model name is correct
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_message
        }
    ]
)

print("Report generation complete.")


Fetching KPI data...
Successfully fetched googleads_analytics.kpi_report_1_summary
Successfully fetched googleads_analytics.kpi_report_2_cpl_ranking
Successfully fetched googleads_analytics.kpi_report_3_top_keywords
Successfully fetched googleads_analytics.kpi_report_4_age_performance
Successfully fetched googleads_analytics.kpi_report_5_gender_performance
Successfully fetched googleads_analytics.kpi_report_6_monthly_trend
Successfully fetched googleads_analytics.kpi_report_7_adgroup_efficiency
Successfully fetched googleads_analytics.kpi_report_8_network_performance
Successfully fetched googleads_analytics.kpi_report_9_day_of_week
Successfully fetched googleads_analytics.kpi_report_10_ad_type_dist
All KPI data fetched.
Sending data to LLM for analysis... This may take a moment.
Report generation complete.


In [0]:
import markdown
# Add CSS wrapper
css_style = """
<style>
table {
  border-collapse: collapse;
  width: 100%;
  margin: 20px 0;
  border: 1px solid #ddd;
}
th, td {
  border: 1px solid #ddd;
  padding: 12px;
  text-align: left;
}
th {
  background-color: #4CAF50;
  color: white;
  font-weight: bold;
}
tr:nth-child(even) {
  background-color: #f2f2f2;
}
tr:hover {
  background-color: #ddd;
}
</style>
"""

html_output = markdown.markdown(
    response.choices[0].message.content[1]['text'], 
    extensions=['tables']
)

# Prepend CSS to HTML
final_html = css_style + html_output

displayHTML(final_html)

print("Report generation complete. The report is rendered above.")

Part 1 – Introduction 
 This report provides a comprehensive analysis of Google Ads account performance for the period July 2024 through March 2026 , utilizing data extracted from the googleads_gold analytical database. 
 The goal is to translate raw campaign metrics into actionable insights and strategic recommendations that improve ad‑spend efficiency and generate qualified leads for the institute’s upcoming data‑engineering training batches in Bangalore and Hyderabad. 
 During the analysed period the account managed eight active campaigns, investing ₹139,608 in advertising. The campaigns generated 1,424,442 impressions, 100,670 clicks and 149 leads (potential student enrollments). The overall click‑through rate was 7.07 percent , the weighted average cost‑per‑click was ₹1.39 , and the cost‑per‑lead was ₹937 . The lead‑to‑click conversion rate was 0.15 percent , indicating that while the ads attract clicks, a small share of that traffic converts into enrollment enquiries. 
 
 Part 2 – Glossary of Key Advertising Metrics 
 
 
 
 Metric 
 Definition 
 Formula (when applicable) 
 
 
 
 
 Impressions 
 Number of times ads were displayed. 
 – 
 
 
 Clicks 
 Number of times ads were clicked. 
 – 
 
 
 CTR (Click‑Through Rate) 
 Percentage of impressions that resulted in clicks. 
 (Clicks ÷ Impressions) × 100 
 
 
 CPC (Cost‑Per‑Click) 
 Average cost paid for each click. 
 Total Cost ÷ Total Clicks 
 
 
 Conversions / Leads 
 Goal completions – form submissions or calls that indicate a potential student enrollment. 
 – 
 
 
 Lead Rate / CVR (Conversion Rate) 
 Percentage of clicks that converted into leads. 
 (Conversions ÷ Clicks) × 100 
 
 
 CPL / CPA (Cost‑Per‑Lead) 
 Average cost to acquire one lead. 
 Total Cost ÷ Total Conversions 
 
 
 
 Industry Benchmarks (India – EdTech – Data‑Engineering focus, Bangalore) 
 
 
 
 Metric 
 Typical Range (India EdTech) 
 Typical Range (Data‑Engineering, Bangalore) 
 Healthy Target Band 
 
 
 
 
 CTR 
 4 – 7 percent (education search) 
 6 – 10 percent with job‑placement hooks 
 6.5 – 8 percent 
 
 
 CPC 
 ₹25 – ₹60 
 ₹40 – ₹80 blended; ₹45 – ₹65 common 
 ₹45 – ₹65 
 
 
 CPL 
 ₹600 – ₹1,500 (qualified leads) 
 ₹800 – ₹1,500 typical 
 ₹1,000 – ₹1,200 (qualified) 
 
 
 Lead Rate / CVR 
 3 – 5 percent (cold), 8 – 12 percent (warm) 
 6 – 10 percent blended likely 
 8 – 10 percent blended (≥10 – 12 percent on warm traffic) 
 
 
 
 Lower CPL and CPC are better; higher CTR and Lead Rate are better. 
 
 Part 3 – Detailed KPI Analysis 
 KPI 1 – High‑Level Performance Summary 
 Objective 
Provide an account‑wide snapshot of spend, reach and lead generation to gauge overall efficiency. 
 Data & Analysis 
 
 
 
 campaign_name 
 campaign_status 
 total_cost_inr 
 total_impressions 
 total_clicks 
 total_leads 
 
 
 
 
 Hyderabad 
 PAUSED 
 33,295 
 244,220 
 10,782 
 4 
 
 
 Azure Data Engineering – Display – Image 
 PAUSED 
 21,791 
 691,883 
 66,109 
 0 
 
 
 Azure Data Engineering – Display – text 
 PAUSED 
 20,056.1 
 20,040 
 398 
 3 
 
 
 Excel in Data Engineering 
 PAUSED 
 18,943.7 
 305,164 
 15,280 
 1 
 
 
 10‑Sept AWS Snowflake 
 PAUSED 
 18,755.6 
 53,903 
 444 
 0 
 
 
 Azure‑Data‑Engineering‑Jan‑2026 
 ENABLED 
 16,707.3 
 13,257 
 582 
 141 
 
 
 Data Engineering (AWS‑Sept) 
 PAUSED 
 6,984.65 
 90,137 
 6,905 
 0 
 
 
 Leads‑Search‑24Feb2025 
 PAUSED 
 3,074.54 
 5,838 
 170 
 0 
 
 
 Account totals 
 – 
 ₹139,608 
 1,424,442 
 100,670 
 149 
 
 
 
 All numbers are taken directly from the high‑level summary table (rows 1–8). 
 The weighted average CPC is ₹1.39 (₹139,608 ÷ 100,670 clicks). The weighted average CPL is ₹937 (₹139,608 ÷ 149 leads). Overall CTR is 7.07 percent (100,670 ÷ 1,424,442 × 100) and the overall lead rate is 0.15 percent (149 ÷ 100,670 × 100). 
 Insights & Recommendations 
 Pros 
- The overall CTR of 7.07 percent exceeds the typical EdTech benchmark of 4 – 7 percent, indicating strong ad relevance. 
- The “Azure‑Data‑Engineering‑Jan‑2026” campaign delivers a CPL of ₹1

Report generation complete. The report is rendered above.
